In [2]:
# Data Preperation
import pandas as pd

df = pd.read_csv('D:/Final Project/Data/news.tsv',sep='\t')
df.head()

,News ID,Category,Topic,Headline,News body,Title entity,Entity content
0,N10000,sports,soccer,Predicting Atlanta United's lineup against Col...,"Only FIVE internationals allowed, count em, FI...","{""Atlanta United's"": 'Atlanta United FC'}","{'Atlanta United FC': {'type': 'item', 'id': '..."
1,N10001,news,newspolitics,Mitch McConnell: DC statehood push is 'full bo...,WASHINGTON -- Senate Majority Leader Mitch McC...,"{'DC': 'Washington, D.C.'}","{'Washington, D.C.': {'type': 'item', 'id': 'Q..."
2,N10002,news,newsus,Home In North Highlands Damaged By Fire,NORTH HIGHLANDS (CBS13) Fire damaged a home ...,{},{}
3,N10003,news,newspolitics,Meghan McCain blames 'liberal media' and 'thir...,Meghan McCain is speaking out after a journali...,{},{}
4,N10004,news,newsworld,Today in History: Aug 1,"1714: George I becomes King Georg Ludwig, Elec...",{},{}


In [3]:
df['Category'].value_counts()

Category
sports           30557
news             26689
finance          10571
lifestyle         7405
autos             5494
travel            5381
foodanddrink      5286
video             4968
tv                3981
health            3799
weather           3298
music             2547
movies            1996
entertainment     1487
kids               299
europe               2
northamerica         1
adexperience         1
Name: count, dtype: int64

In [4]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [5]:
# Data Cleaniing and Preprocessing including Stopword Removal

import re
import string
import pandas as pd
from nltk.corpus import stopwords
import nltk

STOPWORDS = set(stopwords.words('english'))

def clean_text(text):

    if not isinstance(text,str): 
        return ""
    
    text =  re.sub(r"<.*?>"," ",text)

    text = re.sub(r"http\S+|www\S+|https\S+"," ",text)

    text = re.sub(
        "[" 
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
        "]+", 
        "", 
        text
    )

    text = re.sub(r"[^a-zA-Z0-9\s.,!?]", " ", text)

    allowed = set(string.ascii_letters + string.digits + " .,!?")
    text = "".join(ch for ch in text if ch in allowed)

    text = text.lower()

    text = " ".join(text.split())

    tokens = [
        word for word in text.split()
        if word not in STOPWORDS
    ]
    
    text = " ".join(tokens)

    return text

In [6]:
df["text"] = df["Headline"].fillna("") + " " + df["News body"].fillna("")
df = df.rename(columns={"Category": "label"}).dropna()

df["clean_text"] = df["text"].apply(clean_text)

In [7]:
# Label Encoding
from sklearn.preprocessing import LabelEncoder
import joblib
le = LabelEncoder()
df["label_encoded"] = le.fit_transform(df["label"])

In [8]:
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential, Model
import os
from tensorflow.keras.layers import (
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout,
    Input,
    Conv1D,
    GlobalMaxPooling1D,
    Concatenate,
    SpatialDropout1D,       
    BatchNormalization      
)
import pickle

In [9]:
# Parameters

MAX_SEQ_LEN = 200       
VOCAB_SIZE = 50000      
EMBEDDING_DIM = 300     
DROPOUT_RATE = 0.5
CNN_FILTERS = 128
CNN_KERNEL_SIZE = 5
LSTM_UNITS = 128

In [11]:
texts = df["clean_text"].values
labels = df["label"].values

In [12]:
# Encode and Prepare labels

le = LabelEncoder()
y = le.fit_transform(labels)
y = to_categorical(y)
num_classes = y.shape[1]
print("Number of classes:", num_classes)

Number of classes: 18


In [13]:
import gensim.downloader as api

glove_model = api.load("glove-wiki-gigaword-100")

In [16]:
# Train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    texts, y, test_size=0.2, random_state=42
)

In [17]:
# Tokenizer
MAX_WORDS = 50000
MAX_LEN = 150
EMBED_DIM = 300

tokenizer = Tokenizer(num_words=MAX_WORDS,oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

# Convert texts to sequences and pad them
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

# Pad sequences to ensure uniform input size
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post")
X_test_pad  = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post")

vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)

In [18]:
print("Vocab size:", vocab_size)
print("Num classes:", num_classes)

Vocab size: 50000
Num classes: 18


In [19]:
from gensim.scripts.glove2word2vec import glove2word2vec

glove_input = "D:/Final Project/Data/glove.6B/glove.6B.300d.txt"
word2vec_output = "D:/Final Project/Data/glove.6B/glove.6B.300d.w2v.txt"
glove2word2vec(glove_input, word2vec_output)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_27676\1322828516.py:5: DeprecationWarning: Call to deprecated `glove2word2vec` (KeyedVectors.load_word2vec_format(.., binary=False, no_header=True) loads GLoVE text vectors.).
  glove2word2vec(glove_input, word2vec_output)


(400000, 300)

In [21]:
from gensim.models import KeyedVectors

gloVe_model = KeyedVectors.load_word2vec_format(
    r"D:/Final Project/Data/glove.6B/glove.6B.300d.w2v.txt",
    binary=False
)

In [22]:
# Vector Embeddings of GloVe
embedding_matrix_glove = np.zeros((vocab_size,EMBED_DIM))

for word,i in tokenizer.word_index.items():
    if i<MAX_WORDS:
        if word in gloVe_model:
            embedding_matrix_glove[i] = gloVe_model[word]
        else:
            embedding_matrix_glove[i] = np.random.normal(scale=0.6, size=(EMBED_DIM,))

embedding_matrix_glove.shape

(50000, 300)

In [23]:
early = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
    verbose=1
)

BiLSTM + Glove

In [24]:
bilstm_glove = Sequential([
    Embedding(
        vocab_size,
        EMBED_DIM,
        weights=[embedding_matrix_glove],
        input_length=MAX_LEN,
        trainable=False
    ),
    SpatialDropout1D(0.2),        
    Bidirectional(
        LSTM(128, return_sequences=False, dropout=0.3)
    ),
    BatchNormalization(),         
    Dense(256, activation="relu"),
    Dropout(0.4),
    Dense(num_classes, activation="softmax")
])

bilstm_glove.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

print("\nTraining BiLSTM + GloVe")
history3 = bilstm_glove.fit(
    X_train_pad, y_train,
    validation_split=0.2,
    epochs=8,       
    batch_size=128,
    callbacks=[early],
    verbose=1
)

c:\Users\ADMIN\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



Training BiLSTM + GloVe
Epoch 1/8
569/569 ━━━━━━━━━━━━━━━━━━━━ 249s 413ms/step - accuracy: 0.6402 - loss: 1.1508 - val_accuracy: 0.7329 - val_loss: 0.8304
Epoch 2/8
569/569 ━━━━━━━━━━━━━━━━━━━━ 328s 576ms/step - accuracy: 0.7189 - loss: 0.8603 - val_accuracy: 0.7558 - val_loss: 0.7251
Epoch 3/8
569/569 ━━━━━━━━━━━━━━━━━━━━ 369s 649ms/step - accuracy: 0.7443 - loss: 0.7716 - val_accuracy: 0.7683 - val_loss: 0.6946
Epoch 4/8
569/569 ━━━━━━━━━━━━━━━━━━━━ 337s 592ms/step - accuracy: 0.7597 - loss: 0.7246 - val_accuracy: 0.7729 - val_loss: 0.6796
Epoch 5/8
569/569 ━━━━━━━━━━━━━━━━━━━━ 285s 500ms/step - accuracy: 0.7670 - loss: 0.6908 - val_accuracy: 0.7722 - val_loss: 0.6688
Epoch 6/8
569/569 ━━━━━━━━━━━━━━━━━━━━ 206s 362ms/step - accuracy: 0.7742 - loss: 0.6601 - val_accuracy: 0.7798 - val_loss: 0.6510
Epoch 7/8
569/569 ━━━━━━━━━━━━━━━━━━━━ 220s 386ms/step - accuracy: 0.7827 - loss: 0.6324 - val_accuracy: 0.7768 - val_loss: 0.6609
Epoch 8/8
569/569 ━━━━━━━━━━━━━━━━━━━━ 216s 380ms/step - a

TextCNN + Glove

In [25]:
input_layer_g = Input(shape=(MAX_LEN,))
embed_layer_g = Embedding(
    vocab_size,
    EMBED_DIM,
    weights=[embedding_matrix_glove],
    input_length=MAX_LEN,
    trainable=False
)(input_layer_g)

xg = SpatialDropout1D(0.2)(embed_layer_g)

conv3_g = Conv1D(128, 3, activation="relu")(xg)
conv4_g = Conv1D(128, 4, activation="relu")(xg)
conv5_g = Conv1D(128, 5, activation="relu")(xg)

pool3_g = GlobalMaxPooling1D()(conv3_g)
pool4_g = GlobalMaxPooling1D()(conv4_g)
pool5_g = GlobalMaxPooling1D()(conv5_g)

merged_g = Concatenate()([pool3_g, pool4_g, pool5_g])

merged_g = BatchNormalization()(merged_g)

dense_g = Dense(256, activation="relu")(merged_g)
drop_g = Dropout(0.5)(dense_g)
output_layer_g = Dense(num_classes, activation="softmax")(drop_g)

textcnn_glove = Model(inputs=input_layer_g, outputs=output_layer_g)

textcnn_glove.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

print("\n=== Training TextCNN + GloVe ===")
history4 = textcnn_glove.fit(
    X_train_pad, y_train,
    validation_split=0.2,
    epochs=8,
    batch_size=256,
    callbacks=[early],
    verbose=1
)


=== Training TextCNN + GloVe ===
Epoch 1/8
285/285 ━━━━━━━━━━━━━━━━━━━━ 134s 461ms/step - accuracy: 0.6664 - loss: 1.0918 - val_accuracy: 0.7514 - val_loss: 0.7533
Epoch 2/8
285/285 ━━━━━━━━━━━━━━━━━━━━ 125s 440ms/step - accuracy: 0.7437 - loss: 0.7884 - val_accuracy: 0.7687 - val_loss: 0.6997
Epoch 2: early stopping
Restoring model weights from the end of the best epoch: 1.


In [26]:
# save the tokenizer
pickle.dump(tokenizer, open("D:/Final Project/Classify/artifacts/DL/tokenizer.pkl", "wb"))
print("Tokenizer saved")

Tokenizer saved


In [27]:
# EVALUATION AND COMPARISON
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
models = {
    "BiLSTM_GloVe": bilstm_glove,
    "TextCNN_GloVe": textcnn_glove
}

y_test_labels = np.argmax(y_test, axis=1)
dl_results = {}

for name, model in models.items():
    print(f"\nEvaluating {name}...")
    
    y_pred = model.predict(X_test_pad, verbose=0)
    y_pred_labels = np.argmax(y_pred, axis=1)

    acc = accuracy_score(y_test_labels, y_pred_labels)
    
    report = classification_report(
        y_test_labels,
        y_pred_labels,
        output_dict=True,
        zero_division=0
    )

    dl_results[name] = {
        "accuracy": acc,
        "f1_macro": report["macro avg"]["f1-score"],
        "precision_macro": report["macro avg"]["precision"],
        "recall_macro": report["macro avg"]["recall"]
    }

    print(classification_report(
        y_test_labels,
        y_pred_labels,
        zero_division=0
    ))
    print("=" * 80)

df_results = pd.DataFrame(dl_results).T
df_results = df_results.sort_values(by="accuracy", ascending=False)

print("\nDL Model comparison:\n")
print(df_results)

best_dl_model_name = df_results.index[0]
print("\nBest DL model:", best_dl_model_name)
print(df_results.loc[best_dl_model_name])


Evaluating BiLSTM_GloVe...
              precision    recall  f1-score   support

           1       0.85      0.85      0.85      1109
           2       0.51      0.30      0.38       310
           3       0.00      0.00      0.00         1
           4       0.80      0.81      0.80      2070
           5       0.79      0.84      0.81      1034
           6       0.80      0.74      0.77       744
           7       0.00      0.00      0.00        53
           8       0.61      0.67      0.64      1489
           9       0.66      0.68      0.67       415
          10       0.70      0.67      0.68       504
          11       0.77      0.72      0.74      5356
          13       0.93      0.96      0.95      6083
          14       0.65      0.57      0.61      1113
          15       0.62      0.63      0.62       837
          16       0.49      0.58      0.53       969
          17       0.66      0.80      0.73       654

    accuracy                           0.78     2274

In [28]:
# Save DL model
best_dl_model = models[best_dl_model_name]
dl_save_path = f"D:/Final Project/Classify/artifacts/DL/{best_dl_model_name}.h5"
best_dl_model.save(dl_save_path)
print("Model Saved")

Model Saved


In [29]:
import os

df_results = df_results.reset_index().rename(columns={"index": "model_name"})

df_results = df_results.rename(columns={
    "accuracy": "accuracy",
    "precision_macro": "precision",
    "recall_macro": "recall",
    "f1_macro": "f1"
})


df_results = df_results[["model_name", "accuracy", "precision", "recall", "f1"]]


csv_path = "D:/Final Project/Classify/artifacts/model_comparisons.csv"


if os.path.exists(csv_path):
    df_results.to_csv(csv_path, mode="a", index=False, header=False)
else:
    df_results.to_csv(csv_path, index=False)

print(" DL models appended to CSV")
print(df_results)

 DL models appended to CSV
      model_name  accuracy  precision    recall        f1
0   BiLSTM_GloVe  0.777802   0.614836  0.613629  0.611412
1  TextCNN_GloVe  0.747065   0.607571  0.550144  0.564745
